<a href="https://colab.research.google.com/github/GurnoorKaur-30/internship-project_Gurnoor/blob/main/english_and_hindi_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================================
# Hindi -> English translation (fast, NLLB-200-distilled-600M)
# Reads ALL sheets from hindi_dataset_by_section.xlsx, translates each,
# writes translated_by_section.xlsx with the same sheet structure + mt_english column.
#
# Run in Google Colab. Runtime > Change runtime type > GPU (T4).
# Upload hindi_dataset_by_section.xlsx to the Colab session first.
# =========================================================================

!pip install -q transformers sentencepiece accelerate openpyxl

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")

# ---- 1. Load model once, reuse across all sheets ----
MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="hin_Deva")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()
forced_bos = tokenizer.convert_tokens_to_ids("eng_Latn")

def translate_batch(texts, batch_size=32, max_length=512):
    outputs = []
    start = time.time()
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_length=max_length,
                num_beams=1,   # greedy = fastest. num_beams=4 = better quality, ~2-3x slower
            )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(decoded)
        done = min(i + batch_size, len(texts))
        elapsed = time.time() - start
        rate = done / elapsed if elapsed > 0 else 0
        eta = (len(texts) - done) / rate if rate > 0 else 0
        print(f"    {done}/{len(texts)}  ({rate:.1f} rows/sec, ETA {eta/60:.1f} min)")
    return outputs

# ---- 2. Loop through every sheet in the input file ----
INPUT_FILE = "hindi_dataset_by_section.xlsx"
OUTPUT_FILE = "translated_by_section.xlsx"

xl = pd.ExcelFile(INPUT_FILE)
print(f"\nSheets found: {xl.sheet_names}\n")

results = {}
overall_start = time.time()

for sheet in xl.sheet_names:
    df = xl.parse(sheet)
    print(f"--- Translating sheet: {sheet} ({len(df)} rows) ---")
    df["mt_english"] = translate_batch(df["hindi_text"].tolist())
    results[sheet] = df
    print()

print(f"All sheets done in {(time.time()-overall_start)/60:.1f} min total.\n")

# ---- 3. Write all translated sheets back into one multi-sheet file ----
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for sheet, df in results.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

print(f"Saved {OUTPUT_FILE}")

Using device: cuda


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


Sheets found: ['forecast_summary', 'bulletin_subtitle', 'weather_warnings', 'weather_warning_impacts', 'weather_impacts_general', 'general_advisory', 'sms_advisory', 'impact_based_advisories', 'crop_advisory', 'horticulture_advisory', 'livestock_advisory', 'fisheries_advisory', 'poultry_advisory', 'masthead', 'preamble', 'unmatched_other']

--- Translating sheet: forecast_summary (559 rows) ---
    32/559  (5.5 rows/sec, ETA 1.6 min)
    64/559  (8.3 rows/sec, ETA 1.0 min)
    96/559  (9.0 rows/sec, ETA 0.9 min)
    128/559  (10.1 rows/sec, ETA 0.7 min)
    160/559  (9.7 rows/sec, ETA 0.7 min)
    192/559  (8.0 rows/sec, ETA 0.8 min)
    224/559  (7.5 rows/sec, ETA 0.7 min)
    256/559  (7.7 rows/sec, ETA 0.7 min)
    288/559  (8.2 rows/sec, ETA 0.5 min)
    320/559  (8.2 rows/sec, ETA 0.5 min)
    352/559  (8.0 rows/sec, ETA 0.4 min)
    384/559  (7.6 rows/sec, ETA 0.4 min)
    416/559  (7.5 rows/sec, ETA 0.3 min)
    448/559  (7.9 rows/sec, ETA 0.2 min)
    480/559  (8.1 rows/sec, E

In [ ]:
# =========================================================================
# COMBINED PIPELINE: Hindi -> English translation + full validation
# ONE input file, ONE script, ONE run. No file-swapping between steps.
#
# Run in Google Colab. Runtime > Change runtime type > GPU (T4).
# Upload ONLY this file first:
#   - hindi_dataset_by_section_v2.xlsx
# =========================================================================

!pip install -q sacrebleu rouge-score nltk bert-score sentence-transformers unbabel-comet openpyxl
# NOTE: pip will print numpy/protobuf "dependency conflict" warnings above -- these are
# pre-existing Colab package conflicts unrelated to this pipeline. Safe to ignore.

import os
# Force transformers to skip TensorFlow/Flax entirely -- Colab's preinstalled TF version
# conflicts with what bert-score/transformers expects, causing a spurious
# "Backend should be defined in BACKENDS_MAPPING. Offending backend: tf" error.
# Everything in this pipeline runs on PyTorch, so TF/Flax aren't needed at all.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
# Reduces GPU memory fragmentation when loading/unloading multiple large models back to back
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch

def free_gpu():
    """Call after each model is done with -- clears cached GPU memory so the next
    model (which can be several GB) has room, instead of stacking up unfreed memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# =========================================================================
# CONFIG
# =========================================================================
CONFIG = {
    "INPUT_FILE": "hindi_dataset_by_section_v2.xlsx",

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",

    "SOURCE_LANG_NAME": "Hindi",
    "TARGET_LANG_CODE": "en",

    # --- test-first workflow ---
    "TEST_MODE": True,                # True = only run TEST_SECTION. Flip to False once verified.
    "TEST_SECTION": "forecast_summary",

    # --- translation model ---
    "MT_MODEL": "facebook/nllb-200-distilled-600M",
    "MT_BATCH_SIZE": 32,

    # --- toggle metrics on/off ---
    "RUN_BLEU": True,
    "RUN_CHRF": True,
    "RUN_TER": True,
    "RUN_ROUGE_L": True,
    "RUN_METEOR": True,
    "RUN_BERTSCORE": True,
    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",  # lighter than deberta-xlarge-mnli, fits T4 alongside other models
    "BERTSCORE_BATCH_SIZE": 8,
    "RUN_COSINE_LABSE": True,
    "RUN_COMET": True,
    "RUN_COMET_KIWI": True,
    "RUN_ENTITY_NUMERIC_STRUCTURAL": True,

    # --- flagging thresholds ---
    "THRESH_BERTSCORE_F1": 0.85,
    "THRESH_COSINE": 0.75,
    "THRESH_COMET": 0.75,
    "THRESH_CHRF": 50.0,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")


# =========================================================================
# 1. LOAD -- with column checks so a missing file/column fails loudly and clearly
# =========================================================================
def load_data(cfg):
    try:
        xl = pd.ExcelFile(cfg["INPUT_FILE"])
    except FileNotFoundError:
        raise FileNotFoundError(
            f"'{cfg['INPUT_FILE']}' not found in this Colab session. "
            f"Upload it via the folder icon on the left before running this cell."
        )

    sheets = [cfg["TEST_SECTION"]] if cfg["TEST_MODE"] else xl.sheet_names
    missing = [s for s in sheets if s not in xl.sheet_names]
    if missing:
        raise ValueError(f"Sheet(s) {missing} not found. Available sheets: {xl.sheet_names}")

    frames = []
    for sheet in sheets:
        df = xl.parse(sheet)
        required = {cfg["SOURCE_COL"], cfg["REF_COL"], "has_pair", "record_id"}
        missing_cols = required - set(df.columns)
        if missing_cols:
            raise KeyError(
                f"Sheet '{sheet}' is missing columns {missing_cols}. "
                f"Make sure you uploaded hindi_dataset_by_section_v2.xlsx, not an older version. "
                f"Found columns: {list(df.columns)}"
            )
        df["field_name"] = sheet
        df["sheet"] = sheet
        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)
    merged[cfg["SOURCE_COL"]] = merged[cfg["SOURCE_COL"]].astype(str)
    merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)

    print(f"Loaded {len(merged)} rows across sheet(s): {sheets}")
    return merged


# =========================================================================
# 2. TRANSLATE (NLLB-200) -- adds mt_english to the dataframe
# =========================================================================
def translate(df, cfg):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(cfg["MT_MODEL"], src_lang="hin_Deva")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        cfg["MT_MODEL"], torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    model.eval()
    forced_bos = tokenizer.convert_tokens_to_ids("eng_Latn")

    texts = df[cfg["SOURCE_COL"]].tolist()
    outputs = []
    start = time.time()
    batch_size = cfg["MT_BATCH_SIZE"]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=512).to(device)
        with torch.no_grad():
            generated = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                        max_length=512, num_beams=1)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        done = min(i + batch_size, len(texts))
        rate = done / (time.time() - start) if time.time() > start else 0
        print(f"  Translated {done}/{len(texts)}  ({rate:.1f} rows/sec)")

    df[cfg["MT_COL"]] = outputs
    del model, tokenizer
    free_gpu()
    return df


# =========================================================================
# 3. N-GRAM / EDIT-DISTANCE METRICS
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    print("  NLTK data ready.")
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]
    print(f"  Scoring {len(scoreable)} rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...")

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    start = time.time()
    for n, (idx, row) in enumerate(scoreable.iterrows(), 1):
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        if cfg["RUN_BLEU"]:
            bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        if cfg["RUN_CHRF"]:
            chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        if cfg["RUN_TER"]:
            ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        if cfg["RUN_ROUGE_L"]:
            rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        if cfg["RUN_METEOR"]:
            try:
                meteor[idx] = meteor_score([ref.split()], mt.split())
            except Exception:
                meteor[idx] = np.nan
        if n % 50 == 0 or n == len(scoreable):
            rate = n / (time.time() - start) if time.time() > start else 0
            print(f"    {n}/{len(scoreable)}  ({rate:.1f} rows/sec)")

    if cfg["RUN_BLEU"]:
        df["bleu"] = df.index.map(bleu)
    if cfg["RUN_CHRF"]:
        df["chrf++"] = df.index.map(chrf)
    if cfg["RUN_TER"]:
        df["ter"] = df.index.map(ter)
    if cfg["RUN_ROUGE_L"]:
        df["rouge_l"] = df.index.map(rougeL)
    if cfg["RUN_METEOR"]:
        df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 4. BERTSCORE
# =========================================================================
def compute_bertscore(df, cfg):
    from bert_score import score as bertscore
    scoreable = df[df["has_pair"] == True]
    P, R, F1 = bertscore(
        scoreable[cfg["MT_COL"]].tolist(), scoreable[cfg["REF_COL"]].tolist(),
        lang=cfg["TARGET_LANG_CODE"], model_type=cfg["BERTSCORE_MODEL"],
        batch_size=cfg["BERTSCORE_BATCH_SIZE"],
        device=device, verbose=True
    )
    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 5. LaBSE COSINE SIMILARITY
# =========================================================================
def compute_cosine_labse(df, cfg):
    from sentence_transformers import SentenceTransformer, util
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    scoreable = df[df["has_pair"] == True]

    emb_mt = labse.encode(scoreable[cfg["MT_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_ref = labse.encode(scoreable[cfg["REF_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_src = labse.encode(scoreable[cfg["SOURCE_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    del labse
    free_gpu()
    return df


# =========================================================================
# 6. COMET / COMET-KIWI
# =========================================================================
def compute_comet(df, cfg):
    from comet import download_model, load_from_checkpoint
    scoreable = df[df["has_pair"] == True]

    if cfg["RUN_COMET"]:
        print("Loading COMET (reference-based)...")
        path = download_model("Unbabel/wmt22-comet-da")
        model = load_from_checkpoint(path)
        data = [{"src": s, "mt": m, "ref": r} for s, m, r in
                zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])]
        out = model.predict(data, batch_size=8, gpu_ids=[0] if device == "cuda" else None)
        df.loc[scoreable.index, "comet"] = out.scores
        del model
        free_gpu()

    if cfg["RUN_COMET_KIWI"]:
        print("Loading COMET-Kiwi (reference-free)...")
        path = download_model("Unbabel/wmt22-cometkiwi-da")
        model = load_from_checkpoint(path)
        data = [{"src": s, "mt": m} for s, m in zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]])]
        out = model.predict(data, batch_size=8, gpu_ids=[0] if device == "cuda" else None)
        df.loc[scoreable.index, "comet_kiwi"] = out.scores
        del model
        free_gpu()

    return df


# =========================================================================
# 7. ENTITY / NUMERIC / STRUCTURAL CHECKS -- language-agnostic, regex-based
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def segment_count(text, delimiter="/"):
    return len([s for s in text.split(delimiter) if s.strip()])

def segment_count_match(src, mt, delimiter="/"):
    return segment_count(src, delimiter) == segment_count(mt, delimiter)

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)
    df["segment_count_match"] = df.apply(
        lambda r: segment_count_match(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 8. FLAGGING + SUMMARY
# =========================================================================
def flag_and_summarize(df, cfg):
    flags = []
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
        flags.append("flag_low_bertscore")
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
        flags.append("flag_low_cosine")
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
        flags.append("flag_low_comet")
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
        flags.append("flag_low_chrf")
    if "has_untranslated_text" in df:
        flags.append("has_untranslated_text")
    if "segment_count_match" in df:
        df["flag_segment_mismatch"] = ~df["segment_count_match"]
        flags.append("flag_segment_mismatch")

    df["needs_review"] = df[flags].any(axis=1) if flags else False
    print(f"\nFlagged for review: {df['needs_review'].sum()} / {len(df)}")
    return df


# =========================================================================
# MAIN
# =========================================================================
def run_pipeline(cfg):
    start = time.time()
    df = load_data(cfg)

    print("\n--- Translating (Hindi -> English) ---")
    df = translate(df, cfg)

    if cfg["RUN_BLEU"] or cfg["RUN_CHRF"] or cfg["RUN_TER"] or cfg["RUN_ROUGE_L"] or cfg["RUN_METEOR"]:
        print("\n--- N-gram / edit-distance metrics ---")
        df = compute_ngram_metrics(df, cfg)

    if cfg["RUN_BERTSCORE"]:
        print("\n--- BERTScore ---")
        df = compute_bertscore(df, cfg)

    if cfg["RUN_COSINE_LABSE"]:
        print("\n--- LaBSE cosine similarity ---")
        df = compute_cosine_labse(df, cfg)

    if cfg["RUN_COMET"] or cfg["RUN_COMET_KIWI"]:
        print("\n--- COMET / COMET-Kiwi ---")
        df = compute_comet(df, cfg)

    if cfg["RUN_ENTITY_NUMERIC_STRUCTURAL"]:
        print("\n--- Entity / numeric / structural checks ---")
        df = compute_structural_checks(df, cfg)

    df = flag_and_summarize(df, cfg)

    exclude = {cfg["SOURCE_COL"], cfg["MT_COL"], cfg["REF_COL"], "record_id", "field_name",
               "sheet", "has_pair", "state", "district", "month", "date", "row_id"}
    metric_cols = [c for c in df.columns if c not in exclude and df[c].dtype != object]
    scored = df[df["has_pair"] == True]
    summary = scored.groupby("field_name")[metric_cols].mean(numeric_only=True)
    summary["n"] = scored.groupby("field_name").size()

    tag = cfg["TEST_SECTION"] if cfg["TEST_MODE"] else "all_sections"
    out_file = f"validation_results_{tag}.xlsx"
    summary_file = f"validation_summary_{tag}.xlsx"
    df.to_excel(out_file, index=False)
    summary.to_excel(summary_file)

    print(f"\nDone in {(time.time()-start)/60:.1f} min.")
    print(f"Saved {out_file} and {summary_file}")
    print("\n--- Summary ---")
    print(summary)
    return df, summary


# ---- RUN ----
results_df, summary_df = run_pipeline(CONFIG)

Using device: cuda
Loaded 559 rows across sheet(s): ['forecast_summary']

--- Translating (Hindi -> English) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


  Translated 32/559  (8.1 rows/sec)
  Translated 64/559  (11.3 rows/sec)
  Translated 96/559  (11.0 rows/sec)
  Translated 128/559  (12.0 rows/sec)
  Translated 160/559  (11.7 rows/sec)
  Translated 192/559  (11.1 rows/sec)
  Translated 224/559  (11.0 rows/sec)
  Translated 256/559  (10.6 rows/sec)
  Translated 288/559  (11.1 rows/sec)
  Translated 320/559  (10.7 rows/sec)
  Translated 352/559  (10.1 rows/sec)
  Translated 384/559  (9.1 rows/sec)
  Translated 416/559  (9.0 rows/sec)
  Translated 448/559  (9.3 rows/sec)
  Translated 480/559  (9.6 rows/sec)
  Translated 512/559  (9.4 rows/sec)
  Translated 544/559  (9.4 rows/sec)
  Translated 559/559  (9.0 rows/sec)

--- N-gram / edit-distance metrics ---
  NLTK data ready.
  Scoring 481 rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...
    50/481  (3.7 rows/sec)
    100/481  (3.1 rows/sec)
    150/481  (3.3 rows/sec)
    200/481  (3.6 rows/sec)
    250/481  (3.6 rows/sec)
    300/481  (3.5 rows/sec)
    350/481  (3.1 rows/sec)
    400/481  (3.4 

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


TypeError: CometModel.predict() got an unexpected keyword argument 'gpu_ids'

In [ ]:
# =========================================================================
# COMBINED PIPELINE: Hindi -> English translation + full validation
# ONE input file, ONE script, ONE run. No file-swapping between steps.
#
# Run in Google Colab. Runtime > Change runtime type > GPU (T4).
# Upload ONLY this file first:
#   - hindi_dataset_by_section_v2.xlsx
# =========================================================================

!pip install -q sacrebleu rouge-score nltk bert-score sentence-transformers unbabel-comet openpyxl
# NOTE: pip will print numpy/protobuf "dependency conflict" warnings above -- these are
# pre-existing Colab package conflicts unrelated to this pipeline. Safe to ignore.

import os
# Force transformers to skip TensorFlow/Flax entirely -- Colab's preinstalled TF version
# conflicts with what bert-score/transformers expects, causing a spurious
# "Backend should be defined in BACKENDS_MAPPING. Offending backend: tf" error.
# Everything in this pipeline runs on PyTorch, so TF/Flax aren't needed at all.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
# Reduces GPU memory fragmentation when loading/unloading multiple large models back to back
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch

def free_gpu():
    """Call after each model is done with -- clears cached GPU memory so the next
    model (which can be several GB) has room, instead of stacking up unfreed memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# =========================================================================
# CONFIG
# =========================================================================
CONFIG = {
    "INPUT_FILE": "hindi_dataset_by_section_v2.xlsx",

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",

    "SOURCE_LANG_NAME": "Hindi",
    "TARGET_LANG_CODE": "en",

    # --- test-first workflow ---
    "TEST_MODE": True,                # True = only run TEST_SECTION. Flip to False once verified.
    "TEST_SECTION": "forecast_summary",

    # --- translation model ---
    "MT_MODEL": "facebook/nllb-200-distilled-600M",
    "MT_BATCH_SIZE": 32,

    # --- toggle metrics on/off ---
    "RUN_BLEU": True,
    "RUN_CHRF": True,
    "RUN_TER": True,
    "RUN_ROUGE_L": True,
    "RUN_METEOR": True,
    "RUN_BERTSCORE": True,
    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",  # lighter than deberta-xlarge-mnli, fits T4 alongside other models
    "BERTSCORE_BATCH_SIZE": 8,
    "RUN_COSINE_LABSE": True,
    "RUN_COMET": True,
    "RUN_COMET_KIWI": True,
    "RUN_ENTITY_NUMERIC_STRUCTURAL": True,

    # --- flagging thresholds ---
    "THRESH_BERTSCORE_F1": 0.85,
    "THRESH_COSINE": 0.75,
    "THRESH_COMET": 0.75,
    "THRESH_CHRF": 50.0,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")


# =========================================================================
# 1. LOAD -- with column checks so a missing file/column fails loudly and clearly
# =========================================================================
def load_data(cfg):
    try:
        xl = pd.ExcelFile(cfg["INPUT_FILE"])
    except FileNotFoundError:
        raise FileNotFoundError(
            f"'{cfg['INPUT_FILE']}' not found in this Colab session. "
            f"Upload it via the folder icon on the left before running this cell."
        )

    sheets = [cfg["TEST_SECTION"]] if cfg["TEST_MODE"] else xl.sheet_names
    missing = [s for s in sheets if s not in xl.sheet_names]
    if missing:
        raise ValueError(f"Sheet(s) {missing} not found. Available sheets: {xl.sheet_names}")

    frames = []
    for sheet in sheets:
        df = xl.parse(sheet)
        required = {cfg["SOURCE_COL"], cfg["REF_COL"], "has_pair", "record_id"}
        missing_cols = required - set(df.columns)
        if missing_cols:
            raise KeyError(
                f"Sheet '{sheet}' is missing columns {missing_cols}. "
                f"Make sure you uploaded hindi_dataset_by_section_v2.xlsx, not an older version. "
                f"Found columns: {list(df.columns)}"
            )
        df["field_name"] = sheet
        df["sheet"] = sheet
        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)
    merged[cfg["SOURCE_COL"]] = merged[cfg["SOURCE_COL"]].astype(str)
    merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)

    print(f"Loaded {len(merged)} rows across sheet(s): {sheets}")
    return merged


# =========================================================================
# 2. TRANSLATE (NLLB-200) -- adds mt_english to the dataframe
# =========================================================================
def translate(df, cfg):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(cfg["MT_MODEL"], src_lang="hin_Deva")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        cfg["MT_MODEL"], torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    model.eval()
    forced_bos = tokenizer.convert_tokens_to_ids("eng_Latn")

    texts = df[cfg["SOURCE_COL"]].tolist()
    outputs = []
    start = time.time()
    batch_size = cfg["MT_BATCH_SIZE"]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=512).to(device)
        with torch.no_grad():
            generated = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                        max_length=512, num_beams=1)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        done = min(i + batch_size, len(texts))
        rate = done / (time.time() - start) if time.time() > start else 0
        print(f"  Translated {done}/{len(texts)}  ({rate:.1f} rows/sec)")

    df[cfg["MT_COL"]] = outputs
    del model, tokenizer
    free_gpu()
    return df


# =========================================================================
# 3. N-GRAM / EDIT-DISTANCE METRICS
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    print("  NLTK data ready.")
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]
    print(f"  Scoring {len(scoreable)} rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...")

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    start = time.time()
    for n, (idx, row) in enumerate(scoreable.iterrows(), 1):
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        if cfg["RUN_BLEU"]:
            bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        if cfg["RUN_CHRF"]:
            chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        if cfg["RUN_TER"]:
            ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        if cfg["RUN_ROUGE_L"]:
            rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        if cfg["RUN_METEOR"]:
            try:
                meteor[idx] = meteor_score([ref.split()], mt.split())
            except Exception:
                meteor[idx] = np.nan
        if n % 50 == 0 or n == len(scoreable):
            rate = n / (time.time() - start) if time.time() > start else 0
            print(f"    {n}/{len(scoreable)}  ({rate:.1f} rows/sec)")

    if cfg["RUN_BLEU"]:
        df["bleu"] = df.index.map(bleu)
    if cfg["RUN_CHRF"]:
        df["chrf++"] = df.index.map(chrf)
    if cfg["RUN_TER"]:
        df["ter"] = df.index.map(ter)
    if cfg["RUN_ROUGE_L"]:
        df["rouge_l"] = df.index.map(rougeL)
    if cfg["RUN_METEOR"]:
        df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 4. BERTSCORE
# =========================================================================
def compute_bertscore(df, cfg):
    from bert_score import score as bertscore
    scoreable = df[df["has_pair"] == True]
    P, R, F1 = bertscore(
        scoreable[cfg["MT_COL"]].tolist(), scoreable[cfg["REF_COL"]].tolist(),
        lang=cfg["TARGET_LANG_CODE"], model_type=cfg["BERTSCORE_MODEL"],
        batch_size=cfg["BERTSCORE_BATCH_SIZE"],
        device=device, verbose=True
    )
    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 5. LaBSE COSINE SIMILARITY
# =========================================================================
def compute_cosine_labse(df, cfg):
    from sentence_transformers import SentenceTransformer, util
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    scoreable = df[df["has_pair"] == True]

    emb_mt = labse.encode(scoreable[cfg["MT_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_ref = labse.encode(scoreable[cfg["REF_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_src = labse.encode(scoreable[cfg["SOURCE_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    del labse
    free_gpu()
    return df


# =========================================================================
# 6. COMET / COMET-KIWI
# =========================================================================
def compute_comet(df, cfg):
    from comet import download_model, load_from_checkpoint
    scoreable = df[df["has_pair"] == True]

    if cfg["RUN_COMET"]:
        print("Loading COMET (reference-based)...")
        path = download_model("Unbabel/wmt22-comet-da")
        model = load_from_checkpoint(path)
        data = [{"src": s, "mt": m, "ref": r} for s, m, r in
                zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])]
        out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
        df.loc[scoreable.index, "comet"] = out.scores
        del model
        free_gpu()

    if cfg["RUN_COMET_KIWI"]:
        print("Loading COMET-Kiwi (reference-free)...")
        path = download_model("Unbabel/wmt22-cometkiwi-da")
        model = load_from_checkpoint(path)
        data = [{"src": s, "mt": m} for s, m in zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]])]
        out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
        df.loc[scoreable.index, "comet_kiwi"] = out.scores
        del model
        free_gpu()

    return df


# =========================================================================
# 7. ENTITY / NUMERIC / STRUCTURAL CHECKS -- language-agnostic, regex-based
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def segment_count(text, delimiter="/"):
    return len([s for s in text.split(delimiter) if s.strip()])

def segment_count_match(src, mt, delimiter="/"):
    return segment_count(src, delimiter) == segment_count(mt, delimiter)

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)
    df["segment_count_match"] = df.apply(
        lambda r: segment_count_match(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 8. FLAGGING + SUMMARY
# =========================================================================
def flag_and_summarize(df, cfg):
    flags = []
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
        flags.append("flag_low_bertscore")
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
        flags.append("flag_low_cosine")
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
        flags.append("flag_low_comet")
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
        flags.append("flag_low_chrf")
    if "has_untranslated_text" in df:
        flags.append("has_untranslated_text")
    if "segment_count_match" in df:
        df["flag_segment_mismatch"] = ~df["segment_count_match"]
        flags.append("flag_segment_mismatch")

    df["needs_review"] = df[flags].any(axis=1) if flags else False
    print(f"\nFlagged for review: {df['needs_review'].sum()} / {len(df)}")
    return df


# =========================================================================
# MAIN
# =========================================================================
def run_pipeline(cfg):
    start = time.time()
    df = load_data(cfg)

    print("\n--- Translating (Hindi -> English) ---")
    df = translate(df, cfg)

    if cfg["RUN_BLEU"] or cfg["RUN_CHRF"] or cfg["RUN_TER"] or cfg["RUN_ROUGE_L"] or cfg["RUN_METEOR"]:
        print("\n--- N-gram / edit-distance metrics ---")
        df = compute_ngram_metrics(df, cfg)

    if cfg["RUN_BERTSCORE"]:
        print("\n--- BERTScore ---")
        df = compute_bertscore(df, cfg)

    if cfg["RUN_COSINE_LABSE"]:
        print("\n--- LaBSE cosine similarity ---")
        df = compute_cosine_labse(df, cfg)

    if cfg["RUN_COMET"] or cfg["RUN_COMET_KIWI"]:
        print("\n--- COMET / COMET-Kiwi ---")
        df = compute_comet(df, cfg)

    if cfg["RUN_ENTITY_NUMERIC_STRUCTURAL"]:
        print("\n--- Entity / numeric / structural checks ---")
        df = compute_structural_checks(df, cfg)

    df = flag_and_summarize(df, cfg)

    exclude = {cfg["SOURCE_COL"], cfg["MT_COL"], cfg["REF_COL"], "record_id", "field_name",
               "sheet", "has_pair", "state", "district", "month", "date", "row_id"}
    metric_cols = [c for c in df.columns if c not in exclude and df[c].dtype != object]
    scored = df[df["has_pair"] == True]
    summary = scored.groupby("field_name")[metric_cols].mean(numeric_only=True)
    summary["n"] = scored.groupby("field_name").size()

    tag = cfg["TEST_SECTION"] if cfg["TEST_MODE"] else "all_sections"
    out_file = f"validation_results_{tag}.xlsx"
    summary_file = f"validation_summary_{tag}.xlsx"
    df.to_excel(out_file, index=False)
    summary.to_excel(summary_file)

    print(f"\nDone in {(time.time()-start)/60:.1f} min.")
    print(f"Saved {out_file} and {summary_file}")
    print("\n--- Summary ---")
    print(summary)
    return df, summary


# ---- RUN ----
results_df, summary_df = run_pipeline(CONFIG)

Using device: cuda
Loaded 559 rows across sheet(s): ['forecast_summary']

--- Translating (Hindi -> English) ---
  Translated 32/559  (4.1 rows/sec)
  Translated 64/559  (5.3 rows/sec)
  Translated 96/559  (5.6 rows/sec)
  Translated 128/559  (6.5 rows/sec)
  Translated 160/559  (6.8 rows/sec)
  Translated 192/559  (7.1 rows/sec)
  Translated 224/559  (7.4 rows/sec)
  Translated 256/559  (7.6 rows/sec)
  Translated 288/559  (8.0 rows/sec)
  Translated 320/559  (8.1 rows/sec)
  Translated 352/559  (8.0 rows/sec)
  Translated 384/559  (7.5 rows/sec)
  Translated 416/559  (7.5 rows/sec)
  Translated 448/559  (7.8 rows/sec)
  Translated 480/559  (8.0 rows/sec)
  Translated 512/559  (8.0 rows/sec)
  Translated 544/559  (8.1 rows/sec)
  Translated 559/559  (7.8 rows/sec)

--- N-gram / edit-distance metrics ---
  NLTK data ready.
  Scoring 481 rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...
    50/481  (5.2 rows/sec)
    100/481  (5.0 rows/sec)
    150/481  (4.6 rows/sec)
    200/481  (3.9 rows/sec)

  0%|          | 0/121 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/61 [00:00<?, ?it/s]

done in 28.49 seconds, 16.88 sentences/sec

--- LaBSE cosine similarity ---


...

Loading COMET-Kiwi (reference-free)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

KeyError: "Model 'Unbabel/wmt22-cometkiwi-da' not supported by COMET."

In [ ]:
# =========================================================================
# COMBINED PIPELINE: Hindi -> English translation + full validation
# ONE input file, ONE script, ONE run. No file-swapping between steps.
#
# Run in Google Colab. Runtime > Change runtime type > GPU (T4).
# Upload ONLY this file first:
#   - hindi_dataset_by_section_v2.xlsx
# =========================================================================

!pip install -q sacrebleu rouge-score nltk bert-score sentence-transformers unbabel-comet openpyxl
# NOTE: pip will print numpy/protobuf "dependency conflict" warnings above -- these are
# pre-existing Colab package conflicts unrelated to this pipeline. Safe to ignore.

import os
# Force transformers to skip TensorFlow/Flax entirely -- Colab's preinstalled TF version
# conflicts with what bert-score/transformers expects, causing a spurious
# "Backend should be defined in BACKENDS_MAPPING. Offending backend: tf" error.
# Everything in this pipeline runs on PyTorch, so TF/Flax aren't needed at all.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
# Reduces GPU memory fragmentation when loading/unloading multiple large models back to back
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch

def free_gpu():
    """Call after each model is done with -- clears cached GPU memory so the next
    model (which can be several GB) has room, instead of stacking up unfreed memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# =========================================================================
# CONFIG
# =========================================================================
CONFIG = {
    "INPUT_FILE": "hindi_dataset_by_section_v2.xlsx",

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",

    "SOURCE_LANG_NAME": "Hindi",
    "TARGET_LANG_CODE": "en",

    # --- test-first workflow ---
    "TEST_MODE": True,                # True = only run TEST_SECTION. Flip to False once verified.
    "TEST_SECTION": "forecast_summary",

    # --- translation model ---
    "MT_MODEL": "facebook/nllb-200-distilled-600M",
    "MT_BATCH_SIZE": 32,

    # --- toggle metrics on/off ---
    "RUN_BLEU": True,
    "RUN_CHRF": True,
    "RUN_TER": True,
    "RUN_ROUGE_L": True,
    "RUN_METEOR": True,
    "RUN_BERTSCORE": True,
    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",  # lighter than deberta-xlarge-mnli, fits T4 alongside other models
    "BERTSCORE_BATCH_SIZE": 8,
    "RUN_COSINE_LABSE": True,
    "RUN_COMET": True,
    "RUN_COMET_KIWI": False,  # gated model on HF -- needs login + manual license acceptance.
                               # Regular COMET already covers most of the same signal.
                               # See instructions below if you want to enable this later.
    "RUN_ENTITY_NUMERIC_STRUCTURAL": True,

    # --- flagging thresholds ---
    "THRESH_BERTSCORE_F1": 0.85,
    "THRESH_COSINE": 0.75,
    "THRESH_COMET": 0.75,
    "THRESH_CHRF": 50.0,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")


# =========================================================================
# 1. LOAD -- with column checks so a missing file/column fails loudly and clearly
# =========================================================================
def load_data(cfg):
    try:
        xl = pd.ExcelFile(cfg["INPUT_FILE"])
    except FileNotFoundError:
        raise FileNotFoundError(
            f"'{cfg['INPUT_FILE']}' not found in this Colab session. "
            f"Upload it via the folder icon on the left before running this cell."
        )

    sheets = [cfg["TEST_SECTION"]] if cfg["TEST_MODE"] else xl.sheet_names
    missing = [s for s in sheets if s not in xl.sheet_names]
    if missing:
        raise ValueError(f"Sheet(s) {missing} not found. Available sheets: {xl.sheet_names}")

    frames = []
    for sheet in sheets:
        df = xl.parse(sheet)
        required = {cfg["SOURCE_COL"], cfg["REF_COL"], "has_pair", "record_id"}
        missing_cols = required - set(df.columns)
        if missing_cols:
            raise KeyError(
                f"Sheet '{sheet}' is missing columns {missing_cols}. "
                f"Make sure you uploaded hindi_dataset_by_section_v2.xlsx, not an older version. "
                f"Found columns: {list(df.columns)}"
            )
        df["field_name"] = sheet
        df["sheet"] = sheet
        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)
    merged[cfg["SOURCE_COL"]] = merged[cfg["SOURCE_COL"]].astype(str)
    merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)

    print(f"Loaded {len(merged)} rows across sheet(s): {sheets}")
    return merged


# =========================================================================
# 2. TRANSLATE (NLLB-200) -- adds mt_english to the dataframe
# =========================================================================
def translate(df, cfg):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(cfg["MT_MODEL"], src_lang="hin_Deva")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        cfg["MT_MODEL"], torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    model.eval()
    forced_bos = tokenizer.convert_tokens_to_ids("eng_Latn")

    texts = df[cfg["SOURCE_COL"]].tolist()
    outputs = []
    start = time.time()
    batch_size = cfg["MT_BATCH_SIZE"]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=512).to(device)
        with torch.no_grad():
            generated = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                        max_length=512, num_beams=1)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        done = min(i + batch_size, len(texts))
        rate = done / (time.time() - start) if time.time() > start else 0
        print(f"  Translated {done}/{len(texts)}  ({rate:.1f} rows/sec)")

    df[cfg["MT_COL"]] = outputs
    del model, tokenizer
    free_gpu()
    return df


# =========================================================================
# 3. N-GRAM / EDIT-DISTANCE METRICS
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    print("  NLTK data ready.")
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]
    print(f"  Scoring {len(scoreable)} rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...")

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    start = time.time()
    for n, (idx, row) in enumerate(scoreable.iterrows(), 1):
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        if cfg["RUN_BLEU"]:
            bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        if cfg["RUN_CHRF"]:
            chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        if cfg["RUN_TER"]:
            ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        if cfg["RUN_ROUGE_L"]:
            rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        if cfg["RUN_METEOR"]:
            try:
                meteor[idx] = meteor_score([ref.split()], mt.split())
            except Exception:
                meteor[idx] = np.nan
        if n % 50 == 0 or n == len(scoreable):
            rate = n / (time.time() - start) if time.time() > start else 0
            print(f"    {n}/{len(scoreable)}  ({rate:.1f} rows/sec)")

    if cfg["RUN_BLEU"]:
        df["bleu"] = df.index.map(bleu)
    if cfg["RUN_CHRF"]:
        df["chrf++"] = df.index.map(chrf)
    if cfg["RUN_TER"]:
        df["ter"] = df.index.map(ter)
    if cfg["RUN_ROUGE_L"]:
        df["rouge_l"] = df.index.map(rougeL)
    if cfg["RUN_METEOR"]:
        df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 4. BERTSCORE
# =========================================================================
def compute_bertscore(df, cfg):
    from bert_score import score as bertscore
    scoreable = df[df["has_pair"] == True]
    P, R, F1 = bertscore(
        scoreable[cfg["MT_COL"]].tolist(), scoreable[cfg["REF_COL"]].tolist(),
        lang=cfg["TARGET_LANG_CODE"], model_type=cfg["BERTSCORE_MODEL"],
        batch_size=cfg["BERTSCORE_BATCH_SIZE"],
        device=device, verbose=True
    )
    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 5. LaBSE COSINE SIMILARITY
# =========================================================================
def compute_cosine_labse(df, cfg):
    from sentence_transformers import SentenceTransformer, util
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    scoreable = df[df["has_pair"] == True]

    emb_mt = labse.encode(scoreable[cfg["MT_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_ref = labse.encode(scoreable[cfg["REF_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_src = labse.encode(scoreable[cfg["SOURCE_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    del labse
    free_gpu()
    return df


# =========================================================================
# 6. COMET / COMET-KIWI
# =========================================================================
def compute_comet(df, cfg):
    from comet import download_model, load_from_checkpoint
    scoreable = df[df["has_pair"] == True]

    if cfg["RUN_COMET"]:
        print("Loading COMET (reference-based)...")
        path = download_model("Unbabel/wmt22-comet-da")
        model = load_from_checkpoint(path)
        data = [{"src": s, "mt": m, "ref": r} for s, m, r in
                zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])]
        out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
        df.loc[scoreable.index, "comet"] = out.scores
        del model
        free_gpu()

    if cfg["RUN_COMET_KIWI"]:
        try:
            print("Loading COMET-Kiwi (reference-free)...")
            path = download_model("Unbabel/wmt22-cometkiwi-da")
            model = load_from_checkpoint(path)
            data = [{"src": s, "mt": m} for s, m in zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]])]
            out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
            df.loc[scoreable.index, "comet_kiwi"] = out.scores
            del model
            free_gpu()
        except Exception as e:
            print(f"  COMET-Kiwi skipped (likely gated-access error): {e}")
            print("  To enable: log in at huggingface.co, accept the license at "
                  "huggingface.co/Unbabel/wmt22-cometkiwi-da, create a token at "
                  "huggingface.co/settings/tokens, add it as a Colab secret named HF_TOKEN, "
                  "then run: from huggingface_hub import login; login()")

    return df


# =========================================================================
# 7. ENTITY / NUMERIC / STRUCTURAL CHECKS -- language-agnostic, regex-based
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def segment_count(text, delimiter="/"):
    return len([s for s in text.split(delimiter) if s.strip()])

def segment_count_match(src, mt, delimiter="/"):
    return segment_count(src, delimiter) == segment_count(mt, delimiter)

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)
    df["segment_count_match"] = df.apply(
        lambda r: segment_count_match(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 8. FLAGGING + SUMMARY
# =========================================================================
def flag_and_summarize(df, cfg):
    flags = []
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
        flags.append("flag_low_bertscore")
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
        flags.append("flag_low_cosine")
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
        flags.append("flag_low_comet")
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
        flags.append("flag_low_chrf")
    if "has_untranslated_text" in df:
        flags.append("has_untranslated_text")
    if "segment_count_match" in df:
        df["flag_segment_mismatch"] = ~df["segment_count_match"]
        flags.append("flag_segment_mismatch")

    df["needs_review"] = df[flags].any(axis=1) if flags else False
    print(f"\nFlagged for review: {df['needs_review'].sum()} / {len(df)}")
    return df


# =========================================================================
# MAIN
# =========================================================================
def run_pipeline(cfg):
    start = time.time()
    df = load_data(cfg)

    print("\n--- Translating (Hindi -> English) ---")
    df = translate(df, cfg)

    if cfg["RUN_BLEU"] or cfg["RUN_CHRF"] or cfg["RUN_TER"] or cfg["RUN_ROUGE_L"] or cfg["RUN_METEOR"]:
        print("\n--- N-gram / edit-distance metrics ---")
        df = compute_ngram_metrics(df, cfg)

    if cfg["RUN_BERTSCORE"]:
        print("\n--- BERTScore ---")
        df = compute_bertscore(df, cfg)

    if cfg["RUN_COSINE_LABSE"]:
        print("\n--- LaBSE cosine similarity ---")
        df = compute_cosine_labse(df, cfg)

    if cfg["RUN_COMET"] or cfg["RUN_COMET_KIWI"]:
        print("\n--- COMET / COMET-Kiwi ---")
        df = compute_comet(df, cfg)

    if cfg["RUN_ENTITY_NUMERIC_STRUCTURAL"]:
        print("\n--- Entity / numeric / structural checks ---")
        df = compute_structural_checks(df, cfg)

    df = flag_and_summarize(df, cfg)

    exclude = {cfg["SOURCE_COL"], cfg["MT_COL"], cfg["REF_COL"], "record_id", "field_name",
               "sheet", "has_pair", "state", "district", "month", "date", "row_id"}
    metric_cols = [c for c in df.columns if c not in exclude and df[c].dtype != object]
    scored = df[df["has_pair"] == True]
    summary = scored.groupby("field_name")[metric_cols].mean(numeric_only=True)
    summary["n"] = scored.groupby("field_name").size()

    tag = cfg["TEST_SECTION"] if cfg["TEST_MODE"] else "all_sections"
    out_file = f"validation_results_{tag}.xlsx"
    summary_file = f"validation_summary_{tag}.xlsx"
    df.to_excel(out_file, index=False)
    summary.to_excel(summary_file)

    print(f"\nDone in {(time.time()-start)/60:.1f} min.")
    print(f"Saved {out_file} and {summary_file}")
    print("\n--- Summary ---")
    print(summary)
    return df, summary


# ---- RUN ----
results_df, summary_df = run_pipeline(CONFIG)

Using device: cuda
Loaded 559 rows across sheet(s): ['forecast_summary']

--- Translating (Hindi -> English) ---
  Translated 32/559  (7.2 rows/sec)
  Translated 64/559  (10.3 rows/sec)
  Translated 96/559  (10.4 rows/sec)
  Translated 128/559  (11.6 rows/sec)
  Translated 160/559  (11.1 rows/sec)
  Translated 192/559  (10.4 rows/sec)
  Translated 224/559  (10.4 rows/sec)
  Translated 256/559  (10.2 rows/sec)
  Translated 288/559  (10.6 rows/sec)
  Translated 320/559  (10.2 rows/sec)
  Translated 352/559  (9.9 rows/sec)
  Translated 384/559  (8.9 rows/sec)
  Translated 416/559  (8.8 rows/sec)
  Translated 448/559  (9.2 rows/sec)
  Translated 480/559  (9.4 rows/sec)
  Translated 512/559  (9.2 rows/sec)
  Translated 544/559  (9.2 rows/sec)
  Translated 559/559  (8.9 rows/sec)

--- N-gram / edit-distance metrics ---
  NLTK data ready.
  Scoring 481 rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...
    50/481  (4.9 rows/sec)
    100/481  (4.9 rows/sec)
    150/481  (4.5 rows/sec)
    200/481  (4.6 

  0%|          | 0/121 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/61 [00:00<?, ?it/s]

done in 27.35 seconds, 17.59 sentences/sec

--- LaBSE cosine similarity ---


...

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


--- COMET / COMET-Kiwi ---
Loading COMET (reference-based)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment


--- Entity / numeric / structural checks ---

Flagged for review: 509 / 559

Done in 5.7 min.
Saved validation_results_forecast_summary.xlsx and validation_summary_forecast_summary.xlsx

--- Summary ---
                       bleu    chrf++        ter   rouge_l    meteor  \
field_name                                                             
forecast_summary  12.816437  46.50078  69.761851  0.528521  0.376793   

                  bertscore_precision  bertscore_recall  bertscore_f1  \
field_name                                                              
forecast_summary             0.866149          0.791778      0.827011   

                  cosine_mt_vs_ref  cosine_src_vs_ref_direct  ...  \
field_name                                                    ...   
forecast_summary          0.883911                  0.912704  ...   

                  has_untranslated_text  segment_count_match  length_ratio  \
field_name                                                               

In [ ]:
# =========================================================================
# COMBINED PIPELINE v2: Hindi -> English translation + full validation
# ONE input file, ONE script, ONE run. No file-swapping between steps.
#
# WHAT CHANGED FROM v1 (based on forecast_summary test results):
#   1. flag_segment_mismatch was firing on ~90% of rows for no real reason --
#      the "/" delimiter check doesn't apply to prose-style bulletin text.
#      It's now OFF by default (SEGMENT_CHECK_FIELDS controls which fields, if any, use it).
#   2. BERTScore threshold lowered 0.85 -> 0.80 to match observed data
#      (median was 0.838, and COMET/cosine both agreed quality was good at that level).
#   3. BLEU / TER / METEOR / ROUGE-L are now DIAGNOSTIC ONLY -- they are still
#      computed and saved, but no longer drive any flag column. These are surface
#      word-overlap metrics that always look harsh on paraphrastic MT (which NLLB
#      produces), even when translations are semantically correct. Using them as
#      flags was the main reason "everything" looked flagged.
#   4. has_untranslated_text now actually creates a flag column (v1 listed it in
#      flags but never built df["flag_..."] for it).
#   5. The old aggregate "needs_review" column has been REMOVED per request.
#      Individual flag_* columns (flag_low_bertscore, flag_low_cosine, flag_low_comet,
#      flag_low_chrf, flag_untranslated_text, flag_low_numeric_consistency,
#      flag_low_entity_preservation, flag_segment_mismatch) are still saved per-row
#      in the results file so you can filter/sort on whichever ones matter to you.
#   6. Added a "top worst rows" export (worst_rows_<tag>.xlsx) sorted by the
#      metrics most likely to catch real problems, so you can spot-check fast
#      instead of reading through hundreds of rows.
#
# Run in Google Colab. Runtime > Change runtime type > GPU (T4).
# Upload ONLY this file first:
#   - hindi_dataset_by_section_v2.xlsx
# =========================================================================

!pip install -q sacrebleu rouge-score nltk bert-score sentence-transformers unbabel-comet openpyxl
# NOTE: pip will print numpy/protobuf "dependency conflict" warnings above -- these are
# pre-existing Colab package conflicts unrelated to this pipeline. Safe to ignore.

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import time
import pandas as pd
import numpy as np
import torch

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# =========================================================================
# CONFIG
# =========================================================================
CONFIG = {
    "INPUT_FILE": "hindi_dataset_by_section_v2.xlsx",

    "SOURCE_COL": "hindi_text",
    "MT_COL": "mt_english",
    "REF_COL": "english_gold",

    "SOURCE_LANG_NAME": "Hindi",
    "TARGET_LANG_CODE": "en",

    # --- test-first workflow ---
    "TEST_MODE": True,                # True = only run TEST_SECTION. Flip to False once verified.
    "TEST_SECTION": "forecast_summary",

    # --- translation model ---
    "MT_MODEL": "facebook/nllb-200-distilled-600M",
    "MT_BATCH_SIZE": 32,

    # --- toggle metrics on/off (all still computed + saved as diagnostics) ---
    "RUN_BLEU": True,
    "RUN_CHRF": True,
    "RUN_TER": True,
    "RUN_ROUGE_L": True,
    "RUN_METEOR": True,
    "RUN_BERTSCORE": True,
    "BERTSCORE_MODEL": "microsoft/deberta-large-mnli",
    "BERTSCORE_BATCH_SIZE": 8,
    "RUN_COSINE_LABSE": True,
    "RUN_COMET": True,
    "RUN_COMET_KIWI": False,
    "RUN_ENTITY_NUMERIC_STRUCTURAL": True,

    # --- flagging thresholds (recalibrated against forecast_summary results) ---
    "THRESH_BERTSCORE_F1": 0.80,   # was 0.85 -- too strict, median observed was 0.838
    "THRESH_COSINE": 0.75,          # kept -- COMET/cosine already performing well at this bar
    "THRESH_COMET": 0.75,           # kept -- same
    "THRESH_CHRF": 40.0,            # was 50.0 -- lowered toward observed median (~46.5);
                                     # chrF is diagnostic-adjacent but kept as a soft flag
    "THRESH_NUMERIC_CONSISTENCY": 0.5,   # flag rows losing more than half their numbers
    "THRESH_ENTITY_PRESERVATION": 0.4,   # flag rows losing more than 60% of named entities

    # --- which fields (sheet names) actually use "/"-delimited structure and should
    #     get the segment-count check. Leave empty [] to disable entirely (recommended
    #     unless you know specific sections use "/" as a real structural delimiter,
    #     e.g. "18/32" min-max temperature lines). ---
    "SEGMENT_CHECK_FIELDS": [],

    # --- which flag_* columns get computed and printed as a breakdown. BLEU/TER/
    #     METEOR/ROUGE-L deliberately excluded -- they are word-overlap metrics that
    #     score paraphrastic (but correct) MT output harshly, and were responsible
    #     for near-universal false flagging in the v1 run. No aggregate column is
    #     built from these anymore -- each flag_* stays as its own column. ---
    "FLAG_METRICS": [
        "flag_low_bertscore",
        "flag_low_cosine",
        "flag_low_comet",
        "flag_low_chrf",
        "flag_untranslated_text",
        "flag_low_numeric_consistency",
        "flag_low_entity_preservation",
        "flag_segment_mismatch",  # only applies to fields listed in SEGMENT_CHECK_FIELDS
    ],
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")


# =========================================================================
# 1. LOAD
# =========================================================================
def load_data(cfg):
    try:
        xl = pd.ExcelFile(cfg["INPUT_FILE"])
    except FileNotFoundError:
        raise FileNotFoundError(
            f"'{cfg['INPUT_FILE']}' not found in this Colab session. "
            f"Upload it via the folder icon on the left before running this cell."
        )

    sheets = [cfg["TEST_SECTION"]] if cfg["TEST_MODE"] else xl.sheet_names
    missing = [s for s in sheets if s not in xl.sheet_names]
    if missing:
        raise ValueError(f"Sheet(s) {missing} not found. Available sheets: {xl.sheet_names}")

    frames = []
    for sheet in sheets:
        df = xl.parse(sheet)
        required = {cfg["SOURCE_COL"], cfg["REF_COL"], "has_pair", "record_id"}
        missing_cols = required - set(df.columns)
        if missing_cols:
            raise KeyError(
                f"Sheet '{sheet}' is missing columns {missing_cols}. "
                f"Make sure you uploaded hindi_dataset_by_section_v2.xlsx, not an older version. "
                f"Found columns: {list(df.columns)}"
            )
        df["field_name"] = sheet
        df["sheet"] = sheet
        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)
    merged[cfg["SOURCE_COL"]] = merged[cfg["SOURCE_COL"]].astype(str)
    merged[cfg["REF_COL"]] = merged[cfg["REF_COL"]].astype(str)

    print(f"Loaded {len(merged)} rows across sheet(s): {sheets}")
    return merged


# =========================================================================
# 2. TRANSLATE (NLLB-200)
# =========================================================================
def translate(df, cfg):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(cfg["MT_MODEL"], src_lang="hin_Deva")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        cfg["MT_MODEL"], torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    model.eval()
    forced_bos = tokenizer.convert_tokens_to_ids("eng_Latn")

    texts = df[cfg["SOURCE_COL"]].tolist()
    outputs = []
    start = time.time()
    batch_size = cfg["MT_BATCH_SIZE"]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=512).to(device)
        with torch.no_grad():
            generated = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                        max_length=512, num_beams=1)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        done = min(i + batch_size, len(texts))
        rate = done / (time.time() - start) if time.time() > start else 0
        print(f"  Translated {done}/{len(texts)}  ({rate:.1f} rows/sec)")

    df[cfg["MT_COL"]] = outputs
    del model, tokenizer
    free_gpu()
    return df


# =========================================================================
# 3. N-GRAM / EDIT-DISTANCE METRICS (diagnostic only -- not used for flagging)
# =========================================================================
def compute_ngram_metrics(df, cfg):
    import sacrebleu
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    print("  NLTK data ready.")
    from nltk.translate.meteor_score import meteor_score

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scoreable = df[df["has_pair"] == True]
    print(f"  Scoring {len(scoreable)} rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...")

    bleu, chrf, ter, rougeL, meteor = {}, {}, {}, {}, {}
    start = time.time()
    for n, (idx, row) in enumerate(scoreable.iterrows(), 1):
        mt, ref = row[cfg["MT_COL"]], row[cfg["REF_COL"]]
        if cfg["RUN_BLEU"]:
            bleu[idx] = sacrebleu.sentence_bleu(mt, [ref]).score
        if cfg["RUN_CHRF"]:
            chrf[idx] = sacrebleu.sentence_chrf(mt, [ref], word_order=2).score
        if cfg["RUN_TER"]:
            ter[idx] = sacrebleu.sentence_ter(mt, [ref]).score
        if cfg["RUN_ROUGE_L"]:
            rougeL[idx] = rouge.score(ref, mt)["rougeL"].fmeasure
        if cfg["RUN_METEOR"]:
            try:
                meteor[idx] = meteor_score([ref.split()], mt.split())
            except Exception:
                meteor[idx] = np.nan
        if n % 50 == 0 or n == len(scoreable):
            rate = n / (time.time() - start) if time.time() > start else 0
            print(f"    {n}/{len(scoreable)}  ({rate:.1f} rows/sec)")

    if cfg["RUN_BLEU"]:
        df["bleu"] = df.index.map(bleu)
    if cfg["RUN_CHRF"]:
        df["chrf++"] = df.index.map(chrf)
    if cfg["RUN_TER"]:
        df["ter"] = df.index.map(ter)
    if cfg["RUN_ROUGE_L"]:
        df["rouge_l"] = df.index.map(rougeL)
    if cfg["RUN_METEOR"]:
        df["meteor"] = df.index.map(meteor)
    return df


# =========================================================================
# 4. BERTSCORE
# =========================================================================
def compute_bertscore(df, cfg):
    from bert_score import score as bertscore
    scoreable = df[df["has_pair"] == True]
    P, R, F1 = bertscore(
        scoreable[cfg["MT_COL"]].tolist(), scoreable[cfg["REF_COL"]].tolist(),
        lang=cfg["TARGET_LANG_CODE"], model_type=cfg["BERTSCORE_MODEL"],
        batch_size=cfg["BERTSCORE_BATCH_SIZE"],
        device=device, verbose=True
    )
    df.loc[scoreable.index, "bertscore_precision"] = P.numpy()
    df.loc[scoreable.index, "bertscore_recall"] = R.numpy()
    df.loc[scoreable.index, "bertscore_f1"] = F1.numpy()
    free_gpu()
    return df


# =========================================================================
# 5. LaBSE COSINE SIMILARITY
# =========================================================================
def compute_cosine_labse(df, cfg):
    from sentence_transformers import SentenceTransformer, util
    labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    scoreable = df[df["has_pair"] == True]

    emb_mt = labse.encode(scoreable[cfg["MT_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_ref = labse.encode(scoreable[cfg["REF_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    emb_src = labse.encode(scoreable[cfg["SOURCE_COL"]].tolist(), batch_size=32, show_progress_bar=True, convert_to_tensor=True)

    df.loc[scoreable.index, "cosine_mt_vs_ref"] = util.pairwise_cos_sim(emb_mt, emb_ref).cpu().numpy()
    df.loc[scoreable.index, "cosine_src_vs_ref_direct"] = util.pairwise_cos_sim(emb_src, emb_ref).cpu().numpy()
    del labse
    free_gpu()
    return df


# =========================================================================
# 6. COMET / COMET-Kiwi
# =========================================================================
def compute_comet(df, cfg):
    from comet import download_model, load_from_checkpoint
    scoreable = df[df["has_pair"] == True]

    if cfg["RUN_COMET"]:
        print("Loading COMET (reference-based)...")
        path = download_model("Unbabel/wmt22-comet-da")
        model = load_from_checkpoint(path)
        data = [{"src": s, "mt": m, "ref": r} for s, m, r in
                zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]], scoreable[cfg["REF_COL"]])]
        out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
        df.loc[scoreable.index, "comet"] = out.scores
        del model
        free_gpu()

    if cfg["RUN_COMET_KIWI"]:
        try:
            print("Loading COMET-Kiwi (reference-free)...")
            path = download_model("Unbabel/wmt22-cometkiwi-da")
            model = load_from_checkpoint(path)
            data = [{"src": s, "mt": m} for s, m in zip(scoreable[cfg["SOURCE_COL"]], scoreable[cfg["MT_COL"]])]
            out = model.predict(data, batch_size=8, gpus=1 if device == "cuda" else 0)
            df.loc[scoreable.index, "comet_kiwi"] = out.scores
            del model
            free_gpu()
        except Exception as e:
            print(f"  COMET-Kiwi skipped (likely gated-access error): {e}")

    return df


# =========================================================================
# 7. ENTITY / NUMERIC / STRUCTURAL CHECKS -- language-agnostic, regex-based
# =========================================================================
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
CAP_ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z]{2,}\b")

def extract_numbers(text):
    return set(NUM_RE.findall(text))

def numeric_consistency(src, mt):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(mt)) / len(src_nums)

def extract_capitalized_entities(text):
    return set(w.upper() for w in CAP_ENTITY_RE.findall(text))

def entity_preservation(ref, mt):
    ref_ents = extract_capitalized_entities(ref)
    if not ref_ents:
        return 1.0
    return len(ref_ents & extract_capitalized_entities(mt)) / len(ref_ents)

def has_untranslated_source_script(mt_text):
    return bool(DEVANAGARI_RE.search(mt_text))

def segment_count(text, delimiter="/"):
    return len([s for s in text.split(delimiter) if s.strip()])

def segment_count_match(src, mt, delimiter="/"):
    return segment_count(src, delimiter) == segment_count(mt, delimiter)

def length_ratio(src, mt):
    src_len = max(len(src.split()), 1)
    return len(mt.split()) / src_len

def compute_structural_checks(df, cfg):
    df["numeric_consistency"] = df.apply(
        lambda r: numeric_consistency(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    scoreable = df[df["has_pair"] == True]
    df.loc[scoreable.index, "entity_preservation"] = scoreable.apply(
        lambda r: entity_preservation(r[cfg["REF_COL"]], r[cfg["MT_COL"]]), axis=1)
    df["has_untranslated_text"] = df[cfg["MT_COL"]].apply(has_untranslated_source_script)

    # segment_count_match: only computed/applied for fields explicitly listed in
    # SEGMENT_CHECK_FIELDS. For everything else it's set to True (i.e. "not applicable,
    # don't flag") so it can't spuriously fail 90% of rows like in the v1 run.
    if cfg["SEGMENT_CHECK_FIELDS"]:
        apply_mask = df["field_name"].isin(cfg["SEGMENT_CHECK_FIELDS"])
    else:
        apply_mask = pd.Series(False, index=df.index)
    df["segment_count_match"] = True
    df.loc[apply_mask, "segment_count_match"] = df.loc[apply_mask].apply(
        lambda r: segment_count_match(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)

    df["length_ratio"] = df.apply(
        lambda r: length_ratio(r[cfg["SOURCE_COL"]], r[cfg["MT_COL"]]), axis=1)
    return df


# =========================================================================
# 8. FLAGGING + SUMMARY
# =========================================================================
def flag_and_summarize(df, cfg):
    if "bertscore_f1" in df:
        df["flag_low_bertscore"] = df["bertscore_f1"] < cfg["THRESH_BERTSCORE_F1"]
    if "cosine_mt_vs_ref" in df:
        df["flag_low_cosine"] = df["cosine_mt_vs_ref"] < cfg["THRESH_COSINE"]
    if "comet" in df:
        df["flag_low_comet"] = df["comet"] < cfg["THRESH_COMET"]
    if "chrf++" in df:
        df["flag_low_chrf"] = df["chrf++"] < cfg["THRESH_CHRF"]
    if "has_untranslated_text" in df:
        df["flag_untranslated_text"] = df["has_untranslated_text"]
    if "numeric_consistency" in df:
        df["flag_low_numeric_consistency"] = df["numeric_consistency"] < cfg["THRESH_NUMERIC_CONSISTENCY"]
    if "entity_preservation" in df:
        df["flag_low_entity_preservation"] = df["entity_preservation"] < cfg["THRESH_ENTITY_PRESERVATION"]
    if "segment_count_match" in df:
        df["flag_segment_mismatch"] = ~df["segment_count_match"]

    flags = [f for f in cfg["FLAG_METRICS"] if f in df.columns]
    print(f"\nFlag breakdown ({len(df)} rows total):")
    for f in flags:
        print(f"  {f}: {df[f].sum()} rows ({df[f].mean()*100:.1f}%)")
    return df


# =========================================================================
# 9. WORST-ROWS EXPORT -- fastest way to spot-check real problems
# =========================================================================
def export_worst_rows(df, cfg, tag, n=30):
    scoreable = df[df["has_pair"] == True].copy()
    # Rank by a composite of the 3 most reliable quality signals (lower = worse)
    for c in ["comet", "cosine_mt_vs_ref", "bertscore_f1"]:
        if c not in scoreable.columns:
            scoreable[c] = np.nan
    scoreable["quality_rank_score"] = scoreable[["comet", "cosine_mt_vs_ref", "bertscore_f1"]].mean(axis=1)
    worst = scoreable.sort_values("quality_rank_score").head(n)
    cols = [cfg["SOURCE_COL"], cfg["MT_COL"], cfg["REF_COL"], "field_name",
            "comet", "cosine_mt_vs_ref", "bertscore_f1", "chrf++",
            "numeric_consistency", "entity_preservation"]
    cols = [c for c in cols if c in worst.columns]
    out_file = f"worst_rows_{tag}.xlsx"
    worst[cols].to_excel(out_file, index=False)
    print(f"Saved {out_file} ({len(worst)} rows to spot-check)")
    return out_file


# =========================================================================
# MAIN
# =========================================================================
def run_pipeline(cfg):
    start = time.time()
    df = load_data(cfg)

    print("\n--- Translating (Hindi -> English) ---")
    df = translate(df, cfg)

    if cfg["RUN_BLEU"] or cfg["RUN_CHRF"] or cfg["RUN_TER"] or cfg["RUN_ROUGE_L"] or cfg["RUN_METEOR"]:
        print("\n--- N-gram / edit-distance metrics (diagnostic only) ---")
        df = compute_ngram_metrics(df, cfg)

    if cfg["RUN_BERTSCORE"]:
        print("\n--- BERTScore ---")
        df = compute_bertscore(df, cfg)

    if cfg["RUN_COSINE_LABSE"]:
        print("\n--- LaBSE cosine similarity ---")
        df = compute_cosine_labse(df, cfg)

    if cfg["RUN_COMET"] or cfg["RUN_COMET_KIWI"]:
        print("\n--- COMET / COMET-Kiwi ---")
        df = compute_comet(df, cfg)

    if cfg["RUN_ENTITY_NUMERIC_STRUCTURAL"]:
        print("\n--- Entity / numeric / structural checks ---")
        df = compute_structural_checks(df, cfg)

    df = flag_and_summarize(df, cfg)

    exclude = {cfg["SOURCE_COL"], cfg["MT_COL"], cfg["REF_COL"], "record_id", "field_name",
               "sheet", "has_pair", "state", "district", "month", "date", "row_id"}
    metric_cols = [c for c in df.columns if c not in exclude and df[c].dtype != object]
    scored = df[df["has_pair"] == True]
    summary = scored.groupby("field_name")[metric_cols].mean(numeric_only=True)
    summary["n"] = scored.groupby("field_name").size()

    tag = cfg["TEST_SECTION"] if cfg["TEST_MODE"] else "all_sections"
    out_file = f"validation_results_{tag}.xlsx"
    summary_file = f"validation_summary_{tag}.xlsx"
    df.to_excel(out_file, index=False)
    summary.to_excel(summary_file)
    worst_file = export_worst_rows(df, cfg, tag)

    print(f"\nDone in {(time.time()-start)/60:.1f} min.")
    print(f"Saved {out_file}, {summary_file}, {worst_file}")
    print("\n--- Summary ---")
    print(summary)
    return df, summary


# ---- RUN ----
results_df, summary_df = run_pipeline(CONFIG)

Using device: cuda
Loaded 559 rows across sheet(s): ['forecast_summary']

--- Translating (Hindi -> English) ---
  Translated 32/559  (8.6 rows/sec)
  Translated 64/559  (11.8 rows/sec)
  Translated 96/559  (11.4 rows/sec)
  Translated 128/559  (12.1 rows/sec)
  Translated 160/559  (11.8 rows/sec)
  Translated 192/559  (11.2 rows/sec)
  Translated 224/559  (11.1 rows/sec)
  Translated 256/559  (10.6 rows/sec)
  Translated 288/559  (11.1 rows/sec)
  Translated 320/559  (10.7 rows/sec)
  Translated 352/559  (10.1 rows/sec)
  Translated 384/559  (9.1 rows/sec)
  Translated 416/559  (9.0 rows/sec)
  Translated 448/559  (9.3 rows/sec)
  Translated 480/559  (9.5 rows/sec)
  Translated 512/559  (9.4 rows/sec)
  Translated 544/559  (9.3 rows/sec)
  Translated 559/559  (9.0 rows/sec)

--- N-gram / edit-distance metrics (diagnostic only) ---
  NLTK data ready.
  Scoring 481 rows (BLEU/chrF++/TER/ROUGE-L/METEOR)...
    50/481  (4.6 rows/sec)
    100/481  (4.7 rows/sec)
    150/481  (4.4 rows/sec)

  0%|          | 0/121 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/61 [00:00<?, ?it/s]

done in 23.40 seconds, 20.55 sentences/sec

--- LaBSE cosine similarity ---


...

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


--- COMET / COMET-Kiwi ---
Loading COMET (reference-based)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment


--- Entity / numeric / structural checks ---

Flag breakdown (559 rows total):
  flag_low_bertscore: 139 rows (24.9%)
  flag_low_cosine: 0 rows (0.0%)
  flag_low_comet: 5 rows (0.9%)
  flag_low_chrf: 131 rows (23.4%)
  flag_untranslated_text: 0 rows (0.0%)
  flag_low_numeric_consistency: 138 rows (24.7%)
  flag_low_entity_preservation: 96 rows (17.2%)
  flag_segment_mismatch: 0 rows (0.0%)
Saved worst_rows_forecast_summary.xlsx (30 rows to spot-check)

Done in 5.5 min.
Saved validation_results_forecast_summary.xlsx, validation_summary_forecast_summary.xlsx, worst_rows_forecast_summary.xlsx

--- Summary ---
                       bleu    chrf++        ter   rouge_l    meteor  \
field_name                                                             
forecast_summary  12.816437  46.50078  69.761851  0.528521  0.376793   

                  bertscore_precision  bertscore_recall  bertscore_f1  \
field_name                                                              
forecast_summary      